# Run Presidio
Stanza NER (Plain Python)

## Set Up

### Import packages

In [8]:
from pathlib import Path

import pandas as pd
import stanza
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_analyzer.nlp_engine import (
    NerModelConfiguration,
    StanzaNlpEngine,
)

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

### Helper functions

In [9]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def save_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df.to_csv(path, index=False)
        return
    if suffix in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
        return
    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True)
        return
    if suffix == ".json":
        df.to_json(path, orient="records")
        return
    if suffix in {".pkl", ".pickle"}:
        df.to_pickle(path)
        return

    raise ValueError(f"Unsupported output file type: {suffix}")


def normalize_columns(columns):
    if columns is None:
        return []
    if isinstance(columns, str):
        return [col.strip() for col in columns.split(",") if col.strip()]
    return list(columns)


def require_columns(df, columns):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")


def build_stanza_analyzer(
    model_config,
    entity_mapping,
    labels_to_ignore,
    deny_list,
):
    ner_model_configuration = NerModelConfiguration(
        model_to_presidio_entity_mapping=entity_mapping,
        labels_to_ignore=labels_to_ignore,
    )
    stanza_nlp_engine = StanzaNlpEngine(
        models=model_config,
        ner_model_configuration=ner_model_configuration,
    )
    analyzer = AnalyzerEngine(
        nlp_engine=stanza_nlp_engine,
        supported_languages=["en"],
    )

    if deny_list:
        pii_signal_recognizer = PatternRecognizer(
            supported_entity="PII_SIGNAL",
            deny_list=deny_list,
        )
        analyzer.registry.add_recognizer(pii_signal_recognizer)

    return analyzer


def entity_list_with_deny_list(entities, deny_list):
    entities_to_analyze = list(entities)
    if deny_list and "PII_SIGNAL" not in entities_to_analyze:
        entities_to_analyze.append("PII_SIGNAL")
    return entities_to_analyze


def analyze_one_text(analyzer, text, entities, allow_list):
    text = "" if pd.isna(text) else str(text)
    analyzer_results = analyzer.analyze(
        text=text,
        language="en",
        entities=entities,
        allow_list=allow_list,
    )

    detected_entities = ", ".join(
        f"{res.entity_type}: {res.start}-{res.end}" for res in analyzer_results
    )
    flagged_text = ", ".join(
        text[res.start : res.end] for res in analyzer_results
    )

    return {
        "raw_results": str(analyzer_results),
        "detected_entities": detected_entities,
        "flagged_text": flagged_text,
    }


def run_presidio_analysis(df, target_col, analyzer, entities, allow_list):
    require_columns(df, [target_col])
    analysis_rows = [
        analyze_one_text(analyzer, text, entities, allow_list)
        for text in df[target_col]
    ]
    analysis_df = pd.DataFrame(analysis_rows)

    return pd.concat(
        [df.reset_index(drop=True), analysis_df.reset_index(drop=True)],
        axis=1,
    )

### Download Stanza model

In [10]:
# Run once if the English Stanza model is not already installed.
download_stanza_model = False

if download_stanza_model:
    stanza.download("en")

### Set parameters

See available Presidio entity types: https://microsoft.github.io/presidio/supported_entities/

In [18]:
# Replace these paths and column names for your local data.
data_path = Path("data/model_inputs/all_eval_data.csv")
target_col = "text"
id_cols = ["encounter_id", "lab_eventnumber"]
output_path = Path("data/model_outputs/stanza_ner_results.csv")

data = load_dataframe(data_path)
id_col_list = normalize_columns(id_cols)
columns_to_keep = id_col_list + [target_col]
require_columns(data, columns_to_keep)

data = data[columns_to_keep].copy()
display(data.head())
print(f"Loaded {len(data):,} rows")

In [12]:
# # Replace these paths and column names for your local data.
# data_path = Path("data/model_inputs/all_eval_data.csv")
# target_col = "text"
# id_cols = ["hc_id", "encounter_id", "lab_eventnumber"]
# output_path = Path("model_outputs/stanza_ner_results.csv")

# data = load_dataframe(data_path)
# id_col_list = normalize_columns(id_cols)
# columns_to_keep = id_col_list + [target_col]
# require_columns(data, columns_to_keep)

# data = data[columns_to_keep].copy()
# display(data.head())
# print(f"Loaded {len(data):,} rows")


In [13]:
my_entities_list = [
    "PERSON",
    "NRP",
    "DATE_TIME",
    "LOCATION",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "US_SSN",
    "ORGANIZATION",
]
my_allow_list = []
my_deny_list = []

## Set Up Stanza

In [14]:
# Define which model to use.
model_config = [{"lang_code": "en", "model_name": "en"}]

# Define which entities the model returns and how they map to Presidio's.
entity_mapping = {
    "PERSON": "PERSON",
    "NORP": "NRP",
    "FACILITY": "LOCATION",
    "FAC": "LOCATION",
    "ORGANIZATION": "ORGANIZATION",
    "ORG": "LOCATION",
    "GPE": "LOCATION",
    "LOCATION": "LOCATION",
    "DATE": "DATE_TIME",
    "TIME": "DATE_TIME",
}

my_labels_to_ignore = [
    "PRODUCT",
    "EVENT",
    "WORK_OF_ART",
    "LAW",
    "PERCENT",
    "MONEY",
    "QUANTITY",
    "ORDINAL",
    "CARDINAL",
]

## Run Presidio Analyzer

In [15]:
analyzer = build_stanza_analyzer(
    model_config=model_config,
    entity_mapping=entity_mapping,
    labels_to_ignore=my_labels_to_ignore,
    deny_list=my_deny_list,
)
entities_to_analyze = entity_list_with_deny_list(
    my_entities_list,
    my_deny_list,
)

results_df = run_presidio_analysis(
    data,
    target_col=target_col,
    analyzer=analyzer,
    entities=entities_to_analyze,
    allow_list=my_allow_list,
)

## Preview Results

In [16]:
display(results_df.head())
print(f"Output row count: {len(results_df):,}")

## Save Results

In [19]:
save_dataframe(results_df, output_path)
print(f"Saved {len(results_df):,} rows to {output_path.resolve()}")